In [1]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import KNNImputer, SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import recall_score, classification_report, confusion_matrix, precision_score
import numpy as np

In [2]:
df = pd.read_csv('foods_health_scores_allergens.csv')
df

,product_name,brands,categories,ingredients,nutriscore_grade,nova_group,ecoscore_grade,allergens,energy_kcal,fat_100g,...,proteins_100g,salt_100g,sodium_100g,contains_gluten,contains_dairy,contains_nuts,contains_soy,contains_eggs,contains_fish,food_type
0,Sidi Ali,سيدي علي,"en:beverages-and-beverages-preparations, en:be...",OBD1 999 999 1112606 266963207 mb,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000000,0.000000,False,False,False,False,False,False,Branded/Packaged
1,Perly,Perly,"en:dairies, en:fermented-foods, en:fermented-m...","milk cream, cream, sugar, banana, bacteria",UNKNOWN,3.0,B,"en:banana, en:milk",97.0,3.0,...,8.0,NaN,NaN,False,True,False,False,False,False,Branded/Packaged
2,Sidi Ali,Sidi Ali,"en:beverages-and-beverages-preparations, en:be...","Sodium, Calcium, Magnésium, Potassium, Bicarbo...",A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
3,Eau minérale naturelle,sidi ali,"en:beverages-and-beverages-preparations, en:be...",100% mineral water,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,...,NaN,0.065000,0.026000,False,False,False,False,False,False,Branded/Packaged
4,اكوافينا,AQUAFINA,"en:beverages-and-beverages-preparations, en:be...",ouverture et avant le : Voir bouteille. après ...,A,NaN,NOT-APPLICABLE,NaN,0.0,0.0,...,0.0,0.000508,0.000203,False,False,False,False,False,False,Branded/Packaged
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4992,Crème fraîche gastronomique,Président,"en:dairies, en:fermented-foods, en:fermented-m...","_Crème_ (origine France), _ferments lactiques_",D,3.0,A,en:milk,291.0,30.0,...,2.3,0.070000,0.028000,False,True,False,False,False,False,Branded/Packaged
4993,Noix de cajou grillées sans sel,Maître Prunille,"en:plant-based-foods-and-beverages, en:plant-b...",Noix de cajou,B,1.0,E,en:nuts,621.0,47.0,...,21.0,0.020000,0.008000,False,False,True,False,False,False,Branded/Packaged
4994,Cacao puro en polvo desgrasado,Valor,"en:cocoa-and-its-products, en:cocoa-and-chocol...","Cacao desgrasado en polvo, correctores de acid...",C,1.0,F,NaN,375.0,16.0,...,26.0,0.030000,0.012000,False,False,False,False,False,False,Branded/Packaged
4995,Sablés Myrtilles Germes de riz,Gerblé,"en:snacks, en:sweet-snacks, en:biscuits-and-cakes","Farine de blé 58,2%, huile de colza, sucre de ...",A,4.0,C,"en:gluten, en:milk",54.0,2.0,...,0.9,0.050000,0.020000,True,True,False,False,False,False,Branded/Packaged


In [3]:
df["food_type"].value_counts()

food_type
Branded/Packaged    4997
Name: count, dtype: int64

In [4]:
df = df.drop(['product_name', 'brands', 'categories', 'ingredients', 'allergens', 'food_type'], axis = 1)

In [5]:
cols = [
    'contains_gluten',
    'contains_dairy',
    'contains_nuts',
    'contains_soy',
    'contains_eggs',
    'contains_fish'
]

In [6]:
y = df['contains_gluten'].astype(int)

In [7]:
X = df.drop(cols, axis = 1)
X

,nutriscore_grade,nova_group,ecoscore_grade,energy_kcal,fat_100g,saturated_fat_100g,carbs_100g,sugars_100g,fiber_100g,proteins_100g,salt_100g,sodium_100g
0,A,NaN,NOT-APPLICABLE,0.0,0.0,0.0,4.2,1.4,0.0,0.0,0.000000,0.000000
1,UNKNOWN,3.0,B,97.0,3.0,NaN,9.4,NaN,NaN,8.0,NaN,NaN
2,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000
3,A,1.0,NOT-APPLICABLE,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.065000,0.026000
4,A,NaN,NOT-APPLICABLE,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000508,0.000203
...,...,...,...,...,...,...,...,...,...,...,...,...
4992,D,3.0,A,291.0,30.0,22.0,2.9,2.9,NaN,2.3,0.070000,0.028000
4993,B,1.0,E,621.0,47.0,8.7,25.0,6.2,4.5,21.0,0.020000,0.008000
4994,C,1.0,F,375.0,16.0,10.0,16.0,0.7,32.0,26.0,0.030000,0.012000
4995,A,4.0,C,54.0,2.0,0.2,7.9,1.9,0.5,0.9,0.050000,0.020000


In [8]:
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=0.2,        
    random_state=5831,      
    stratify=y           
)

In [9]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.25,        
    random_state=5831,      
    stratify=y           
)

In [10]:
baseline_recall = (y_test == 1).mean()
print("Baseline recall:", baseline_recall)

Baseline recall: 0.32


In [11]:
# show numeric and categorical columns
num_cols = ['energy_kcal','fat_100g', 'saturated_fat_100g', 'carbs_100g', 'sugars_100g','fiber_100g', 'proteins_100g', 'salt_100g', 'sodium_100g']     
cat_cols = ['nutriscore_grade', 'nova_group', 'ecoscore_grade']     

# pipeline for numeric
num_pipeline = Pipeline([
    ("imputer", KNNImputer(n_neighbors=5)),
    ("scaler", StandardScaler())
])

# pipeline for categorical
cat_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# combine preprocessing
preprocess = ColumnTransformer([
    ("num", num_pipeline, num_cols),
    ("cat", cat_pipeline, cat_cols)
])

# full pipeline
pipe = Pipeline([
    ("preprocess", preprocess),
    ("lda", LinearDiscriminantAnalysis())
])

In [12]:
pipe.fit(X_train, y_train)
y_prob = pipe.predict_proba(X_val)[:, 1]

In [13]:
thresholds = np.arange(0.05, 0.96, 0.05)
recalls = dict()
for t in thresholds:
    y_pred = (y_prob >= t).astype(int)
    if precision_score(y_val, y_pred, pos_label=1) > 0.5:
        recalls[t] = recall_score(y_val, y_pred, pos_label = 1)

max_key = max(recalls, key=recalls.get)
print(max_key)
print(recalls[max_key])

0.15000000000000002
0.92


In [14]:
y_prob = pipe.predict_proba(X_test)[:, 1]

In [15]:
y_pred = (y_prob >= max_key).astype(int)

In [16]:
recall = recall_score(y_test, y_pred, pos_label=1)
print("Recall:", recall)
precision = precision_score(y_test, y_pred, pos_label=1)
print("Precision:", precision)

Recall: 0.91875
Precision: 0.5547169811320755


In [17]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual 0", "Actual 1"],
    columns=["Pred 0", "Pred 1"]
)

print(cm_df)

          Pred 0  Pred 1
Actual 0     444     236
Actual 1      26     294


In [18]:
feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
coefs = pipe.named_steps["lda"].coef_[0]

coef_df = pd.DataFrame({
    "feature": feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs)
}).sort_values("abs_coefficient", ascending=False)

print(coef_df)

                                 feature  coefficient  abs_coefficient
3                        num__carbs_100g     2.651836         2.651836
1                          num__fat_100g    -1.945056         1.945056
0                       num__energy_kcal     1.870338         1.870338
27    cat__ecoscore_grade_NOT-APPLICABLE    -1.586256         1.586256
15         cat__nutriscore_grade_UNKNOWN    -1.547105         1.547105
4                       num__sugars_100g    -1.329860         1.329860
14  cat__nutriscore_grade_NOT-APPLICABLE    -1.209359         1.209359
16                   cat__nova_group_1.0    -0.798293         0.798293
13               cat__nutriscore_grade_E     0.783639         0.783639
2                num__saturated_fat_100g    -0.592542         0.592542
19                   cat__nova_group_4.0     0.506686         0.506686
5                        num__fiber_100g    -0.402924         0.402924
21            cat__ecoscore_grade_A-PLUS    -0.364425         0.364425
6     

## New stuff starts here!

In [27]:
def process(df, y_col: str, cols):
    y = df[y_col].astype(int)
    X = df.drop(cols, axis=1)

    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=5831,
        stratify=y
    )

    X_train, X_val, y_train, y_val = train_test_split(
        X_trainval, y_trainval,
        test_size=0.25,
        random_state=5831,
        stratify=y_trainval
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

def get_recalls(thresholds, y_prob, y_val):
    recalls = dict()
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        if precision_score(y_val, y_pred, pos_label=1) > 0.5:
            recalls[t] = recall_score(y_val, y_pred, pos_label = 1)
        
    max_key = max(recalls, key=recalls.get)
    return recalls, max_key

def evaluate_on_test_set(pipe, X_test, y_test, max_key):
    y_prob = pipe.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= max_key)
    
    recall = recall_score(y_test, y_pred, pos_label=1)
    precision = precision_score(y_test, y_pred, pos_label=1)
    
    cm = confusion_matrix(y_test, y_pred)

    cm_df = pd.DataFrame(
        cm,
        index=["Actual 0", "Actual 1"],
        columns=["Pred 0", "Pred 1"]
    )

    return precision, recall, cm_df

def get_coef_df(pipe):
    feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
    coefs = pipe.named_steps["lda"].coef_[0]

    coef_df = pd.DataFrame({
        "feature": feature_names,
        "coefficient": coefs,
        "abs_coefficient": np.abs(coefs)
    }).sort_values("abs_coefficient", ascending=False)

    return coef_df
    

In [28]:
gluten_X_train, gluten_X_val, gluten_X_test, gluten_y_train, gluten_y_val, gluten_y_test = process(df, 'contains_gluten', cols)

baseline_recall_gluten = (gluten_y_test == 1).mean()
print("Baseline recall for gluten:", baseline_recall_gluten)

Baseline recall for gluten: 0.32


In [29]:
pipe.fit(gluten_X_train, gluten_y_train)
gluten_y_prob = pipe.predict_proba(gluten_X_val)[:, 1]

In [30]:
gluten_recalls, gluten_max_key = get_recalls(thresholds, gluten_y_prob, gluten_y_val)
print(gluten_max_key)
print(gluten_recalls[gluten_max_key])

0.15000000000000002
0.953125


In [31]:
gluten_test_precision, gluten_test_recall, gluten_cm_df = evaluate_on_test_set(pipe, gluten_X_test, gluten_y_test, gluten_max_key)
print("Precision:", gluten_test_precision)
print("Recall:", gluten_test_recall)

Precision: 0.5431192660550459
Recall: 0.925


In [32]:
print(gluten_cm_df)

          Pred 0  Pred 1
Actual 0     431     249
Actual 1      24     296


In [33]:
gluten_coef_df = get_coef_df(pipe)
print(coef_df)

                                 feature  coefficient  abs_coefficient
3                        num__carbs_100g     2.651836         2.651836
1                          num__fat_100g    -1.945056         1.945056
0                       num__energy_kcal     1.870338         1.870338
27    cat__ecoscore_grade_NOT-APPLICABLE    -1.586256         1.586256
15         cat__nutriscore_grade_UNKNOWN    -1.547105         1.547105
4                       num__sugars_100g    -1.329860         1.329860
14  cat__nutriscore_grade_NOT-APPLICABLE    -1.209359         1.209359
16                   cat__nova_group_1.0    -0.798293         0.798293
13               cat__nutriscore_grade_E     0.783639         0.783639
2                num__saturated_fat_100g    -0.592542         0.592542
19                   cat__nova_group_4.0     0.506686         0.506686
5                        num__fiber_100g    -0.402924         0.402924
21            cat__ecoscore_grade_A-PLUS    -0.364425         0.364425
6     